In [1]:
!pip install -q pytorch-lightning datasets transformers evaluate accelerate scikit-learn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 852.4/852.4 kB 23.4 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 9.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.4/983.4 kB 65.5 MB/s eta 0:00:00


In [2]:
# Instalacion de librerias
import random
import torch
import numpy as np
import os
from pytorch_lightning import seed_everything
import matplotlib.pyplot as plt
import seaborn as sns
import re

seed_val = 42
random.seed(seed_val)
np.random.seed(seed_val)
torch.manual_seed(seed_val)
torch.cuda.manual_seed_all(seed_val)# Store the average loss after eachepoch so we can plot them.
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
os.environ["TF_DETERMINISTIC_OPS"] = "1" # See:https://github.com/NVIDIA/tensorflow-determinism#confirmed-current-gpu-specific-sources-of-non-determinism-with-solutions
seed_everything(42, workers=True)

from datasets import Dataset, DatasetDict #, load_metric EN PRINCIPIO ESTÁ DESCONTINUADO, TENDREMOS QUE BUSCAR OTRA ALTERNATIVA
import pandas as pd
import sklearn as sk
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, average_precision_score, f1_score
from sklearn.model_selection import train_test_split
from transformers import AutoTokenizer, AutoModelForSequenceClassification, \
TrainingArguments, Trainer, pipeline, EarlyStoppingCallback
from huggingface_hub import login

INFO:lightning_fabric.utilities.seed:Seed set to 42


In [3]:
# Comprobación de disponibilidad de GPU
if torch.cuda.is_available():
    # Si hay una GPU disponible, la asignamos como dispositivo principal
    device = torch.device("cuda")
    print(f'✅ GPU detectada. Trabajando con: "{torch.cuda.get_device_name(0)}"')
else:
    # Si no hay GPU, el modelo y los tensores irán a la CPU
    device = torch.device("cpu")
    print('⚠️ Trabajando con CPU.')
    print('Para usar la GPU en Colab: Ve al menú superior "Entorno de ejecución" -> "Cambiar tipo de entorno de ejecución" -> Selecciona "GPU T4".')

# Opcional pero recomendado: Ver cuánta memoria VRAM tienes disponible
if device.type == 'cuda':
    memoria_total = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'Memoria VRAM total disponible: {memoria_total:.2f} GB')

✅ GPU detectada. Trabajando con: "Tesla T4"
Memoria VRAM total disponible: 15.64 GB


Montura de google drive para poder acceder a los datasets

In [5]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


# Lectura y etiquetado de Datos

In [6]:
# ================================
# 1. Carga y preparación de datos
# ================================
# Cargar dataset de entrenamiento con etiquetas
#df = pd.read_csv("/home/adrian/Escritorio/DeepSexist/DatasetManagement/EXIST2025DatasetV0.3/EXIST 2025 Videos Dataset/training/EXIST2025_training_task3_1_DISCARD.csv")
df = pd.read_csv("/content/drive/MyDrive/dataset/EXIST 2025 Videos Dataset/training/EXIST2025_training_3_1.csv")
df = df.rename(columns={"label_task_3_1_merged": "label"})
df = df.drop(columns=["Unnamed: 0"])
df["label"] = df["label"].astype(int)

# from sklearn.model_selection import train_test_split
# train_df, eval_df = train_test_split(df, test_size=0.15, stratify=df["label"], random_state=42)

# train_dataset = Dataset.from_pandas(train_df)
# eval_dataset = Dataset.from_pandas(eval_df)

# PENDIENTE DE DEFINIR UN FICHERO TRAIN, VAL Y TEST
# División estratificada
train_df, temp_df = train_test_split(df, test_size=0.3, stratify=df["label"], random_state=42)
val_df, test_df = train_test_split(temp_df, test_size=0.5, stratify=temp_df['label'], random_state=42)

print("Distribución fichero de entrenamiento:")
print(train_df.value_counts('label'))

print("Distribución fichero de test:")
print(test_df.value_counts('label'))

print("Distribución fichero de validación:")
print(val_df.value_counts('label'))


Distribución fichero de entrenamiento:
label
0    914
1    841
Name: count, dtype: int64
Distribución fichero de test:
label
0    196
1    181
Name: count, dtype: int64
Distribución fichero de validación:
label
0    196
1    180
Name: count, dtype: int64


# Preprocesado del texto

**¿Es realmente necesario preprocesar una transcripción?**

In [7]:
# Para ver el vocabulario del dataset

import pandas as pd

# Suponiendo que la columna de texto se llama 'text'
# Tokeniza y obtiene todas las palabras
vocabulario = set(
    palabra.lower()
    for fila in df["text"].dropna()
    for palabra in fila.split()
)

# Mostrar todo el vocabulario
print(vocabulario)

# (Opcional) Ver el número de palabras únicas
print(f"\nTotal de palabras únicas: {len(vocabulario)}")


{'steam', 'sleepovers', 'rap,', 'screaming?', 'jajajajajjajajajajajaj', 'wondrous', 'haces', 'pagarla', 'brinley.', 'threat.', 'conocí,', 'liberals', 'llamas', '.urazamuleres', 'julián.', 'cars', 'banda', 'problemas', 'think,', 'jefa', 'jail!', 'arieta', 'depa.', 'minuto', 'testimonio', 'perú,', 'festejamos', 'avión', 'worshipping', 'salón,', 'eduxrdovarq', 'announced', 'cop.', 'mermaids?', 'ambición,', 'salvo.', 'supe', 'cheese.', 'man!!!', "it'.", 'tucl', 'experts', 'cleaned', 'appreciation', '#cocinajoven', 'chanclas.', 'women,', 'demostramos.', "'benaye,", 'medicine', 'failed', 'diario.', 'lado,', 'i4e', '¿es', 'made', '#justiceforsarah.', '(22-a2)', 'p.m.', 'nalgas.', 'towels.', 'espera;', 'lives?', 'categoría.y', 'cuenten', 'house', 'peor', 'abrázalos', 'av-025', 'fuerte', 'sufrir', 'seguía,', 'gllyn', '{anis', 'vivamos', "€'", 'frustrated', 'ov', 'realizan', 'magdalena,', 'tomaron', '_toxic774d.', 'entitlement', 'transgénero', 'abegita', 'mandato?', 'invalidar', '4l', 'balik', '

In [8]:
import re

def remove_links(tweet):
    """Takes a string and removes web links from it"""
    tweet = re.sub(r'http\S+', '', tweet)        # remove http links
    tweet = re.sub(r'bit.ly/\S+', '', tweet)     # remove bitly links
    tweet = re.sub(r'\[link\]', '', tweet )      # remove [link]
    tweet = re.sub(r'\[url\]', '', tweet )       # remove [url]
    tweet = re.sub(r'pic.twitter\S+','', tweet)
    return tweet

def remove_users(tweet):
    """Takes a string and removes retweet and @user information"""
    tweet = re.sub('(RT\s@[A-Za-z]+[A-Za-z0-9-_]+)', '', tweet)  # remove re-tweet
    tweet = re.sub('(@[A-Za-z]+[A-Za-z0-9-_]+)', '', tweet)      # remove tweeted at
    tweet = re.sub(r'\[user\]', '', tweet )                      # remove [user]
    return tweet

def remove_hashtags(tweet):
    """Takes a string and removes any hash tags"""
    tweet = re.sub('(#[A-Za-z]+[A-Za-z0-9-_]+)', '', tweet)      # remove hash tags
    return tweet

def remove_av(tweet):
    """Takes a string and removes AUDIO/VIDEO tags or labels"""
    tweet = re.sub('VIDEO:', '', tweet)  # remove 'VIDEO:' from start of tweet
    tweet = re.sub('AUDIO:', '', tweet)  # remove 'AUDIO:' from start of tweet
    return tweet

def remove_emojis(tweet):
    emoj = re.compile("["
        u"\U00002700-\U000027BF"  # Dingbats
        u"\U0001F600-\U0001F64F"  # Emoticons
        u"\U00002600-\U000026FF"  # Miscellaneous Symbols
        u"\U0001F300-\U0001F5FF"  # Miscellaneous Symbols And Pictographs
        u"\U0001F900-\U0001F9FF"  # Supplemental Symbols and Pictographs
        u"\U0001FA70-\U0001FAFF"  # Symbols and Pictographs Extended-A
        u"\U00010000-\U0010FFFF"
        u"\U0001F680-\U0001F6FF"  # Transport and Map Symbols
        u"\U0001F1E0-\U0001F1FF"  # flags (iOS)
        u"\U00002702-\U000027B0"
        u"\U000024C2-\U0001F251"
        u"\U00002702-\U000027B0"
        u"\U000024C2-\U0001F251"
        u"\U0001f926-\U0001f937"
        u"\U00010000-\U0010ffff"
        u"\u2640-\u2642"
        u"\u2600-\u2B55"
        u"\ufe0f"  # dingbats

                      "]+", re.UNICODE)
    return re.sub(emoj, '', tweet)

<>:14: SyntaxWarning: invalid escape sequence '\s'
<>:14: SyntaxWarning: invalid escape sequence '\s'
/tmp/ipykernel_1155/1710960392.py:14: SyntaxWarning: invalid escape sequence '\s'
  tweet = re.sub('(RT\s@[A-Za-z]+[A-Za-z0-9-_]+)', '', tweet)  # remove re-tweet


Convertimos todo a minúsculas

In [9]:
campo_texto = 'text'

train_df[campo_texto] = train_df[campo_texto].str.lower()
val_df[campo_texto] = val_df[campo_texto].str.lower()
test_df[campo_texto] = test_df[campo_texto].str.lower()

# Definición de métricas

In [10]:
# Función para realizar distintas métricas en ejecución

def compute_metrics(eval_pred):

  ##############
  ## predictions son logits, que son tuplas de la forma [valor1, valor2]
  ## Por ejemplo [-1.5606991,  1.6122842] significa que ha predicho eso para un documento
  ## Eso es lo que pasa a la última capa del transformer (softmax si es binario)
  ## Por eso se utiliza el índice del valor máximo de la tupla, para decir que esa es la clase que predice

  ## label_ids = [0, 1, 1, 0, 1]  # Etiquetas reales
  ## predictions = [
  ##  [0.8, 0.2],  # Predicciones para la primera instancia
  ##  [0.3, 0.7],  # Predicciones para la segunda instancia
  ##  [0.1, 0.9],  # Predicciones para la tercera instancia
  ##  [0.9, 0.1],  # Predicciones para la cuarta instancia
  ##  [0.4, 0.6],  # Predicciones para la quinta instancia
  ##           ]

  ##############

  labels = eval_pred.label_ids
  preds = eval_pred.predictions.argmax(-1)

  # Compute precision, recall, F1-score, and support
  precision, recall, f1, _ = sk.metrics.precision_recall_fscore_support(labels, preds, average="macro")

  # Calculate F1-score for the minority class (label = 1)
  f1_minoritaria= f1_score(labels, preds, pos_label=1)

  # Calculate F1-score for the majority class (label = 0)
  f1_mayoritaria = f1_score(labels, preds, pos_label=0)

  # Calculate accuracy
  acc = sk.metrics.accuracy_score(labels, preds)

  # Calculate Area Under the Curve (AUC)
  AUC = roc_auc_score(labels, preds)

  # Calculate Precision-Recall Area Under the Curve (AUC)
  PREC_REC = average_precision_score(labels, preds)

  return {
      'accuracy': acc,
      'f1': f1,
      'precision': precision,
      'recall': recall,
      'AUC': AUC,
      'f1_minoritaria': f1_minoritaria,
      'f1_mayoritaria': f1_mayoritaria,
      'PREC_REC': PREC_REC
  }

# Entrenamiento del modelo

In [19]:
# Cargamos el token de HuggingFace que lo tenemos en un fichero oculto e iniciamos sesión

#with open("/home/adrian/Escritorio/DeepSexist/DatasetManagement/EXIST2025DatasetV0.3/huggingfaceToken.txt", "r") as file:
#    hf_token = file.read().strip()

from huggingface_hub import login
#from google.colab import userdata

#hf_token = userdata.get('HF_TOKEN')

ruta_archivo = '/content/drive/MyDrive/dataset/EXIST 2025 Videos Dataset/token_hf.txt'

try:
    # Abrimos el archivo en modo lectura ('r)
    with open(ruta_archivo, 'r') as file:
        hf_token = file.read().strip()

        # Hacemos el login
        login(token=hf_token)
        print("Login completado con éxito")
except:
    print("Error: No se ha encontrado el archivo " + ruta_archivo + " en la ruta especificada")

#login(hf_token)

Login completado con éxito


In [11]:
# Eleccion del modelo
#model_checkpoint = "xlm-roberta-base"
#model_checkpoint = 'bert-base-multilingual-uncased'
#model_checkpoint = 'annahaz/xlm-roberta-base-misogyny-sexism-indomain-mix-bal'
model_checkpoint = 'jhu-clsp/mmBERT-base'

tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)

config.json:   0%|          | 0.00/1.19k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/46.4k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.5M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/636 [00:00<?, ?B/s]

In [12]:
# Se carga el modelo preentrenado
n_labels = 2

# El uso de una función de inicialización facilita la repetición del entrenamiento
# Se puede usar la misma función de inicialización en diferentes ejecuciones del código o en configuraciones de entrenamiento diferentes
# Esto facilita la repetición del entrenamiento y la reproducibilidad, ya que se puede inicializar el modelo
# de la misma manera en cada ejecución.

def model_init():
    return AutoModelForSequenceClassification.from_pretrained(model_checkpoint,
                                                              num_labels = n_labels) #, return_dict = True )
                                                              # use_auth_token = 'token propio de HugginFace')

In [13]:
# Para saber el nombre del modelo
model_name = model_checkpoint.split("/")[-1]
model_name

'mmBERT-base'

## Fine-tuning

In [14]:
# Selección de hiperparámetros
#BATCH_SIZE = 32
BATCH_SIZE = 16
NUM_TRAIN_EPOCHS = 15
#LEARNING_RATE = 3e-5
LEARNING_RATE = 1e-5
MAX_LENGTH = 128
WEIGHT_DECAY = 0.1

In [15]:
def tokenize_data(examples):
  return tokenizer(examples[campo_texto], truncation=True, max_length=MAX_LENGTH, padding=True)

In [18]:
from transformers import Trainer, TrainingArguments

optim = ["adamw_hf", "adamw_torch", "adamw_apex_fused", "adafactor", "adamw_torch_xla"]
MAX_LENGTH = 128

training_args = TrainingArguments(
    output_dir = 'results',
    num_train_epochs = NUM_TRAIN_EPOCHS,
    learning_rate = LEARNING_RATE,
    per_device_train_batch_size = BATCH_SIZE,
    per_device_eval_batch_size = BATCH_SIZE,
    load_best_model_at_end = True,
    metric_for_best_model = 'f1', # Cambiar la metrica por la que queremos ajustar
    weight_decay = WEIGHT_DECAY,
    eval_strategy = 'epoch',
    save_strategy = 'epoch',
    save_total_limit = 3,
    optim = optim[1],
    push_to_hub = False,
    greater_is_better = True,
    logging_strategy = 'epoch'
)

#output_dir = '/home/adrian/Escritorio/DeepSexist/TrainingBooks/Task3.1/Research/ModelosTransformers'
output_dir = '/content/drive/MyDrive/dataset/EXIST 2025 Videos Dataset/training/Research/Transformers/mmBERT-base'

# Entrenamiento de cada modelo por separado
model_output_dir = f"{output_dir}/modelo_{model_checkpoint}"

# Convierte directamente los DataFrames de Pandas a Datasets de Hugging Face
train_dataset = Dataset.from_pandas(train_df)
valid_dataset = Dataset.from_pandas(val_df)

# Reseteamos el formato para que no haya fallos
train_dataset.reset_format()
valid_dataset.reset_format()

# Obtenemos los nombres de todas las columnas originales
columns_train = train_dataset.column_names
columns_valid = valid_dataset.column_names

#Protegemos la columna "label" para no eliminarla accidentalmente

columna_etiqueta = "label" if "label" in columns_train else "labels"

if columna_etiqueta in columns_train:
    columns_train.remove(columna_etiqueta)
if columna_etiqueta in columns_valid:
    columns_valid.remove(columna_etiqueta)

encoded_train_dataset = train_dataset.map(tokenize_data, batched=True, remove_columns=columns_train)
encoded_valid_dataset = valid_dataset.map(tokenize_data, batched=True, remove_columns=columns_valid)

# Redefinimos model_init aquí para aplicar la congelación (Layer Freezing)
# def model_init():
#     # 1. Cargamos el modelo base
#     model = AutoModelForSequenceClassification.from_pretrained(model_checkpoint, num_labels=n_labels)
#     # 2. Congelamos todas las capas base (el "cerebro")
#     for param in model.base_model.parameters():
#         param.requires_grad = False
#     # El classifier sigue estando activo por defecto
#     return model

# Descongelación parcial de las últimas 3 capas

def model_init():
    # 1. Cargamos el modelo base
    model = AutoModelForSequenceClassification.from_pretrained(model_checkpoint, num_labels=n_labels)

    # 2. Congelamos TODO el cerebro base por defecto
    for param in model.base_model.parameters():
        param.requires_grad = False

    # 3. Buscamos dónde guarda el modelo sus capas (esto varía según el modelo que estemos trabajando)
    if hasattr(model.base_model, 'encoder') and hasattr(model.base_model.encoder, 'layer'):
        capas_transformers = model.base_model.encoder.layer
    elif hasattr(model.base_model, 'layers'):
        capas_transformers = model.base_model.layers
    else:
        print("No se encontró la estructura de capas estándar. Se entrenará solo con la cabeza.")
        capas_transformers=[]

    # 4. Descongelamos solo las últimas 3 capas para que se adapten al vocabulario (TikTok)
    capas_a_descongelar = 3
    if len(capas_transformers) >= capas_a_descongelar:
        print("Descongelando las últimas " + str(capas_a_descongelar) + " capas de un total de " + str(len(capas_transformers)))
        for capa in capas_transformers[-capas_a_descongelar:]:
            for param in capa.parameters():
                param.requires_grad = True
    return model

# Inicializa el entrenador
trainer = Trainer(
    model_init = model_init,
    args=training_args,
    compute_metrics = compute_metrics,
    train_dataset=encoded_train_dataset,
    eval_dataset=encoded_valid_dataset,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)],
    processing_class=tokenizer,
)

# Entrena el modelo
trainer.train()

# Guarda el modelo entrenado
trainer.save_model(model_output_dir)

Map:   0%|          | 0/1755 [00:00<?, ? examples/s]

Map:   0%|          | 0/376 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/136 [00:00<?, ?it/s]

[transformers] ModernBertForSequenceClassification LOAD REPORT from: jhu-clsp/mmBERT-base
Key               | Status     | 
------------------+------------+-
decoder.bias      | UNEXPECTED | 
decoder.weight    | UNEXPECTED | 
classifier.weight | MISSING    | 
classifier.bias   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Descongelando las últimas 3 capas de un total de 22


...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

In [ ]:
# Convertimos a Dataset de HuggingFace
train_dataset = Dataset.from_pandas(train_df)
valid_dataset = Dataset.from_pandas(val_df)

# Reseteamos el formato por si viene con formato anterior
train_dataset.reset_format()
valid_dataset.reset_format()

# Eliminamos columnas innecesarias excepto "labels" y texto (ej. "text")
columns_train = train_dataset.column_names
columns_valid = valid_dataset.column_names
columns_train.remove("label")
columns_valid.remove("label")

# Tokenizamos los datasets
encoded_train_dataset = train_dataset.map(tokenize_data, batched=True, remove_columns=columns_train)
encoded_valid_dataset = valid_dataset.map(tokenize_data, batched=True, remove_columns=columns_valid)

# Hiperparámetros
training_args = TrainingArguments(
    output_dir='results',
    num_train_epochs=NUM_TRAIN_EPOCHS,
    learning_rate=LEARNING_RATE,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    load_best_model_at_end=True,
    metric_for_best_model='recall',  # Puedes usar 'accuracy' o 'f1'
    weight_decay=WEIGHT_DECAY,
    evaluation_strategy='epoch',
    save_strategy='epoch',
    save_total_limit=3,
    optim="adamw_torch",  # Puedes cambiarlo por otro optimizador si prefieres
    push_to_hub=False
)

# Entrenador
trainer = Trainer(
    model_init=model_init,
    args=training_args,
    compute_metrics=compute_metrics,
    train_dataset=encoded_train_dataset,
    eval_dataset=encoded_valid_dataset,
    tokenizer=tokenizer,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)]
)

# Entrenamiento
trainer.train()

# Guardar el modelo
output_dir = '/home/adrian/Escritorio/DeepSexist/TrainingBooks/Task3.1/Research/ModelosTransformers' + model_checkpoint
trainer.save_model(output_dir)

Map:   0%|          | 0/1755 [00:00<?, ? examples/s]

Map:   0%|          | 0/376 [00:00<?, ? examples/s]

/home/adrian/anaconda3/lib/python3.12/site-packages/transformers/training_args.py:1559: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(
/tmp/ipykernel_3208/4199679674.py:37: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(
Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-multilingual-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-multilingual-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be ab

  0%|          | 0/825 [00:00<?, ?it/s]

  0%|          | 0/12 [00:00<?, ?it/s]

...

{'eval_loss': 0.8155430555343628, 'eval_accuracy': 0.6728723404255319, 'eval_f1': 0.6728144830952735, 'eval_precision': 0.6751812366737739, 'eval_recall': 0.6746598639455783, 'eval_AUC': 0.6746598639455783, 'eval_f1_minoritaria': 0.6771653543307087, 'eval_f1_mayoritaria': 0.6684636118598383, 'eval_PREC_REC': 0.5955885466285593, 'eval_runtime': 0.7293, 'eval_samples_per_second': 515.573, 'eval_steps_per_second': 16.454, 'epoch': 4.0}


  0%|          | 0/12 [00:00<?, ?it/s]

{'eval_loss': 1.032497525215149, 'eval_accuracy': 0.6702127659574468, 'eval_f1': 0.6699793312381437, 'eval_precision': 0.6700056593095642, 'eval_recall': 0.670294784580499, 'eval_AUC': 0.6702947845804988, 'eval_f1_minoritaria': 0.6612021857923497, 'eval_f1_mayoritaria': 0.6787564766839378, 'eval_PREC_REC': 0.5942207478583594, 'eval_runtime': 0.7266, 'eval_samples_per_second': 517.495, 'eval_steps_per_second': 16.516, 'epoch': 5.0}


  0%|          | 0/12 [00:00<?, ?it/s]

{'eval_loss': 1.358781337738037, 'eval_accuracy': 0.6835106382978723, 'eval_f1': 0.6835083996463307, 'eval_precision': 0.6845587193653492, 'eval_recall': 0.6846371882086169, 'eval_AUC': 0.6846371882086167, 'eval_f1_minoritaria': 0.6826666666666666, 'eval_f1_mayoritaria': 0.6843501326259946, 'eval_PREC_REC': 0.6050784991210524, 'eval_runtime': 0.7284, 'eval_samples_per_second': 516.194, 'eval_steps_per_second': 16.474, 'epoch': 6.0}
{'train_runtime': 83.1999, 'train_samples_per_second': 316.407, 'train_steps_per_second': 9.916, 'train_loss': 0.41995789498993846, 'epoch': 6.0}
